# Python Clean Code Practices

This single notebook consolidates all five modules of the *Clean Code Practices* path into
one runnable, self-study document. Every concept is paired with executable examples so you
can experiment as you read.

## What you'll learn

| # | Module | Focus |
|---|--------|-------|
| 1 | Writing Maintainable Python Code | Quality attributes, KISS, YAGNI, SRP, cohesion, coupling, DRY, naming |
| 2 | Python Style Guidelines & PEP 8 | Layout, imports, naming, tooling (ruff, black) |
| 3 | Documenting Your Project | Docstrings (PEP 257), Google/NumPy styles, Sphinx |
| 4 | Error Handling & Exceptions | try/except, EAFP, custom exceptions, layering, `raise from` |
| 5 | Effective Code Reviews | The code review pyramid, giving constructive feedback |

> **Why clean code?** Code is read far more often than it is written. Clean code is
> code that a teammate — or *you*, six months from now — can read, understand, and change
> safely. The reference book for this discipline is *Clean Code* by Robert C. Martin.

---

## Part 1 — Writing Maintainable Python Code

### What makes code "good"?

Good code satisfies a set of **quality attributes**. Correctness is the price of entry;
these attributes are what separate code that merely works from code that keeps working:

| Attribute | Question it answers |
|-----------|---------------------|
| **Readability** | Can another person understand it quickly? |
| **Maintainability** | How easily can it be changed or fixed? |
| **Extensibility** | Can new features be added without rewrites? |
| **Testability** | Can behavior be verified in isolation? |
| **Performance** | Does it meet its speed/resource budget? |

Many of these pull in the same direction: code that is readable is usually easier to
maintain, extend, and test.

### General Best Practices

These are broad heuristics that apply to almost any piece of code.

- **KISS — Keep It Short and Simple.** The simplest solution that works is usually the best.
- **The Principle of Least Surprise.** Code should behave the way a reader expects. A function named `get_user` should not also delete the user.
- **YAGNI — You Aren't Gonna Need It.** Don't add features, flexibility, or abstraction "just in case." Don't show off. Build what is required now.
- **Avoid premature optimization.** Don't complicate code for speed you haven't measured. Bottlenecks are hard to predict — you're probably wasting your time.

> On performance: **measure first.** When a real performance problem appears, measure
> *where* it occurs, then fix the actual bottleneck. Guessing leads to complex code that
> optimizes the wrong thing.

### YAGNI in action

Below, two solutions compute the average grade per student. The first is a plain function.
The second is a full Strategy-pattern class hierarchy — flexible, extensible... and complete
overkill if all you were asked for is the average.

Building the elaborate version *before* you need multiple statistics is a YAGNI violation.
Run both and notice they produce the same answer for the task actually at hand.

In [1]:
students = [
    {"name": "Alice", "grades": [85, 90, 78]},
    {"name": "Bob", "grades": [92, 88, 95]},
    {"name": "Charlie", "grades": [70, 75, 80]},
    {"name": "Diana", "grades": [88, 82, 91]},
]


# The simple solution — this is all the task asked for.
def average(students: list[dict]) -> dict[str, float]:
    results = {}
    for student in students:
        grades = student["grades"]
        results[student["name"]] = sum(grades) / len(grades) if grades else 0.0
    return results


print(average(students))

{'Alice': 84.33333333333333, 'Bob': 91.66666666666667, 'Charlie': 75.0, 'Diana': 87.0}


In [2]:
from abc import ABC, abstractmethod


# The "just in case" solution — a Strategy hierarchy.
# Only justified once you ACTUALLY need median, top grade, etc.
class CalculationStrategy(ABC):
    @abstractmethod
    def calculate(self, grades: list[float]) -> float:
        """Calculate a statistic from a list of grades."""
        raise NotImplementedError


class AverageGradeStrategy(CalculationStrategy):
    def calculate(self, grades: list[float]) -> float:
        return sum(grades) / len(grades) if grades else 0.0


class StatisticsCalculator:
    def __init__(self, strategy: CalculationStrategy):
        self.strategy = strategy

    def calculate(self, students: list[dict]) -> dict[str, float]:
        return {s["name"]: self.strategy.calculate(s["grades"]) for s in students}


calculator = StatisticsCalculator(AverageGradeStrategy())
print(calculator.calculate(students))
print("Same result — but a lot more machinery. Add it only when a second strategy is real.")

{'Alice': 84.33333333333333, 'Bob': 91.66666666666667, 'Charlie': 75.0, 'Diana': 87.0}
Same result — but a lot more machinery. Add it only when a second strategy is real.


### Code Structure: Cohesion & Coupling

Three related ideas govern how you split a program into parts. They apply to **every**
level of your code — functions, classes, modules, and packages.

- **Single Responsibility Principle (SRP).** Every component should have responsibility over a single part of the program's functionality — one you can explain in a short sentence. If your sentence needs an "and," you probably have two responsibilities.
- **High Cohesion.** The degree to which the elements of a component belong together. A cohesive component has a small, well-defined purpose and does as little as possible.
- **Loose Coupling.** How much one component knows about the inner workings of another. We want components to be *minimally dependent* on each other.

> **The goal: high cohesion, loose coupling.** You should be able to change the
> implementation details of one class without breaking the classes that depend on it.

### Tight coupling — the problem

Here `Broker.total_value` reaches *inside* each `Portfolio` and reads its `position`
dictionary directly. The broker now knows how a portfolio stores its holdings. The moment
`Portfolio` changes that internal structure, the broker breaks.

In [3]:
current_prices = {"AAPL": 100, "MSFT": 80, "GOOGL": 90}


class Portfolio:
    def __init__(self):
        self.position = {}

    def buy(self, symbol, amount):
        self.position[symbol] = self.position.get(symbol, 0) + amount


class Broker:
    def __init__(self):
        self.portfolio_list: list[Portfolio] = []

    def new_portfolio(self) -> Portfolio:
        p = Portfolio()
        self.portfolio_list.append(p)
        return p

    def total_value(self):
        # Tightly coupled: depends on Portfolio's internal `position` dict.
        total = 0
        for portfolio in self.portfolio_list:
            for symbol, amount in portfolio.position.items():
                total += current_prices[symbol] * amount
        return total


b = Broker()
b.new_portfolio().buy("AAPL", 10)
p2 = b.new_portfolio()
p2.buy("MSFT", 5)
print("Total value (tight):", b.total_value())

Total value (tight): 1400


### Loose coupling — the fix

Give `Portfolio` a `value()` method and let the broker simply **ask** each portfolio for
its value. The broker no longer knows or cares how a portfolio is stored internally. This
is loose coupling: each class owns its own details.

In [4]:
class Portfolio:
    def __init__(self):
        self.position = {}

    def buy(self, symbol, amount):
        self.position[symbol] = self.position.get(symbol, 0) + amount

    def value(self):
        """Portfolio knows how to value ITSELF."""
        return sum(current_prices[s] * amt for s, amt in self.position.items())


class Broker:
    def __init__(self):
        self.portfolio_list: list[Portfolio] = []

    def new_portfolio(self) -> Portfolio:
        p = Portfolio()
        self.portfolio_list.append(p)
        return p

    def total_value(self):
        # Loosely coupled: just asks each portfolio for its value.
        return sum(p.value() for p in self.portfolio_list)


b = Broker()
b.new_portfolio().buy("AAPL", 10)
b.new_portfolio().buy("GOOGL", 8)
print("Total value (loose):", b.total_value())

Total value (loose): 1720


### Separation of Concerns & DRY

- **Separation of Concerns.** Split your system into parts that each address one well-defined responsibility. Classic examples: **HTML / CSS / JavaScript** (structure / style / behavior) and **Model / View / Controller** (data / presentation / control flow).
- **DRY — Don't Repeat Yourself.** Every piece of knowledge should have a single, authoritative representation. Duplicated logic means duplicated bugs and duplicated fixes.

The counterweight to DRY is *not* WET but good judgment: extract a shared abstraction only
when the duplication represents the *same* concept, not two things that merely look alike
today.

### Naming

Names are the primary interface to your code. Good names remove the need for comments.

- Names should be **clear and descriptive**.
- If you can't think of a clear, short name, **think again about the component's responsibility** — a naming struggle often signals a design problem (a violation of SRP).
- Use **nouns** for classes and objects (`Invoice`, `user`).
- Use **verbs** for functions (`calculate_total`, `send_email`).

**Python naming standard (from PEP 8):**

| Kind | Convention | Example |
|------|-----------|---------|
| Modules | short, lowercase | `billing`, `user_mgr` |
| Classes | `CapitalizedNaming` | `InvoiceItem` |
| Functions / variables | `lowercase_with_underscores` | `total_amount` |
| Constants | `ALL_CAPS` | `MAX_RETRIES` |
| Non-public names | leading underscore | `_internal_cache` |

In [5]:
# Naming demo — read these aloud; the names carry the meaning.
MAX_LOGIN_ATTEMPTS = 3          # constant: ALL_CAPS


class LoginThrottle:            # class: noun, CapitalizedNaming
    def __init__(self):
        self._attempts = 0      # non-public: leading underscore

    def register_failure(self):  # function: verb, lowercase_with_underscores
        self._attempts += 1

    def is_locked(self) -> bool:
        return self._attempts >= MAX_LOGIN_ATTEMPTS


throttle = LoginThrottle()
for _ in range(4):
    throttle.register_failure()
print("Locked out:", throttle.is_locked())

Locked out: True


### Functions

Small functions are easier to read, test, and reuse.

- **Keep them short.** A function should fit on your screen. A common guideline is a maximum of ~20 lines (opinions vary).
- **Don't indent too many levels.** Deep nesting is a sign the function does too much; extract helpers or use early returns.
- **Limit arguments.** Many parameters make a function hard to call correctly.
- **Use type hints.** They document intent and enable tooling.
- **Document** the function with a docstring.
- **Return a single type.** A function that sometimes returns a `Student`, sometimes a `list`, and sometimes `None` forces every caller to write defensive code.

### Return a single type

The first `get_students` below returns `list | Student | None` — three shapes. Every caller
must check which one they got. The second version *always* returns a list (possibly empty).
Callers can iterate unconditionally.

In [6]:
class Student:
    def __init__(self, name: str):
        self.name = name

    def __repr__(self) -> str:
        return f"Student({self.name})"


def run_query() -> list[Student]:
    return [Student("Alice"), Student("Bob"), Student("Charlie")]


# Inconsistent: list | Student | None  ->  callers must branch on the type.
def get_students_bad(found: list[Student]):
    if not found:
        return None
    if len(found) == 1:
        return found[0]
    return found


# Better: ALWAYS return a list.
def get_students_good(found: list[Student]) -> list[Student]:
    return list(found)


print("bad, 0 results ->", get_students_bad([]))
print("bad, 1 result  ->", get_students_bad(run_query()[:1]))
print("good, 0 results ->", get_students_good([]))
print("good, 1 result  ->", get_students_good(run_query()[:1]))
# With the "good" version, callers can always write:  for s in get_students_good(...)

bad, 0 results -> None
bad, 1 result  -> Student(Alice)
good, 0 results -> []
good, 1 result  -> [Student(Alice)]


### Classes

The same principles apply, plus a few class-specific ones:

- **Single responsibility**, **cohesive**, with **descriptive names**.
- Provide **documentation** and respect **encapsulation** (hide internal state; expose behavior).
- **Use `dataclasses`** for classes that mainly represent data — each object is a distinct value.
- **Prefer immutable classes.** Use `@dataclass(frozen=True)`, or expose read-only `@property` values on regular classes. Immutable objects are easier to reason about and safe to share.

In [7]:
from dataclasses import dataclass
from datetime import date


@dataclass
class InvoiceItem:
    description: str
    quantity: int
    unit_price: float

    @property
    def total(self) -> float:
        return self.quantity * self.unit_price


@dataclass(frozen=True)          # immutable: an invoice never changes once issued
class Invoice:
    invoice_number: str
    customer_name: str
    issue_date: date
    items: list[InvoiceItem]

    @property
    def subtotal(self) -> float:
        return sum(item.total for item in self.items)

    @property
    def tax_amount(self) -> float:
        return self.subtotal * 0.21          # 21% VAT

    @property
    def total_amount(self) -> float:
        return self.subtotal + self.tax_amount


invoice = Invoice(
    invoice_number="2026-001",
    customer_name="Acme Corp",
    issue_date=date(2026, 9, 29),
    items=[InvoiceItem("Widget", 3, 10.0), InvoiceItem("Gadget", 1, 25.0)],
)
print(f"Subtotal: {invoice.subtotal:.2f}")
print(f"Tax:      {invoice.tax_amount:.2f}")
print(f"Total:    {invoice.total_amount:.2f}")

# Immutability check: this raises FrozenInstanceError.
try:
    invoice.customer_name = "Someone Else"
except Exception as e:
    print(f"Cannot mutate frozen invoice: {type(e).__name__}")

Subtotal: 55.00
Tax:      11.55
Total:    66.55
Cannot mutate frozen invoice: FrozenInstanceError


---
## Part 2 — Following Python Style Guidelines: PEP 8

### What is a PEP?

A **PEP** (Python Enhancement Proposal) is a design document that provides information to
the Python community or describes a new feature for Python, its processes, or its
environment.

### PEP 8 — the style guide

**PEP 8 is about *style*, not correctness.** Code that ignores PEP 8 can still run
perfectly. But a consistent style makes a codebase readable and lets a whole team work in
one another's files without friction. PEP 8 covers:

- **Layout** of code and comments (formatting)
- **Whitespace and punctuation**
- **Naming**

> *"A foolish consistency is the hobgoblin of little minds"* — Ralph Waldo Emerson, quoted
> in PEP 8 itself. The point: follow the guide, but know when a rule hurts readability in a
> specific case. Style serves the reader, not the other way around.

### Layout rules

| Rule | Value |
|------|-------|
| Indentation | **4 spaces** per level (never tabs) |
| Max line length | **79 characters** |
| Blank lines between top-level functions/classes | **2** |
| Blank lines between methods inside a class | **1** |
| Spaces in expressions | around operators, after commas |

The two cells below show the same code laid out badly and then correctly.

In [8]:
# NON-compliant layout (still runs, but hard to read)
def add(a,b):return a+b
class calc:
    def double( self,x ):
        return x*2
    def triple(self,x):
        return x*3
print(add(2,3), calc().double(4))

5 8


In [9]:
# PEP 8-compliant layout
def add(a, b):
    return a + b


class Calc:
    def double(self, x):
        return x * 2

    def triple(self, x):
        return x * 3


print(add(2, 3), Calc().double(4))

5 8


### Imports

- Imports should be on **separate lines**.
- Group them into **three blocks separated by blank lines**, in this order:
  1. **Standard library** (`os`, `sys`, `json`)
  2. **Third-party libraries** (`requests`, `numpy`)
  3. **Local application / library** imports (`from myproject import ...`)

```python
# 1. Standard library
import os
import sys
from pathlib import Path

# 2. Third-party
import requests
from rich import print

# 3. Local application
from myproject.billing import charge_new_user
from myproject.db import DB
```

Avoid `from module import *` — it pollutes the namespace and hides where names come from.

### Naming (PEP 8 recap)

Identical to the naming table in Part 1 — because good naming *is* a PEP 8 concern:

- **Modules:** short, lowercase names
- **Classes:** `CapitalizedNaming`
- **Functions:** `lowercase_with_underscores`
- **Constants:** `ALL_CAPS`
- **Non-public names:** start with an underscore

### Documentation

PEP 8 says to write **docstrings for all public modules, functions, classes, and methods.**
The detailed conventions for *how* to write those docstrings live in **PEP 257** — the
subject of Part 3.

### Checking your code automatically

You don't have to police PEP 8 by hand. Tools check and even fix problems for you:

| Tool | Role |
|------|------|
| **ruff** | Extremely fast linter + formatter (Rust-based); a modern all-in-one |
| **black** | Opinionated autoformatter — reformats to a canonical style |
| **flake8** | Classic linter |
| **pylint** | Deep static analysis and style checks |

**Where to run them:**

- **In CI/CD** — fail the build if code doesn't meet quality standards.
- **As a Git pre-commit hook** — catch issues before they're ever committed.

```bash
# Typical local workflow
ruff check .        # report problems
ruff check --fix .  # auto-fix what it safely can
ruff format .       # reformat (black-compatible)
```

> **If you get whitespace/punctuation wrong, your tools will help you.** That's the whole
> point — let the machine handle mechanical consistency so humans can focus on design.

In [10]:
# A tiny "linter" to make the idea concrete: flag lines over 79 characters.
sample = [
    "x = 1",
    "message = 'this line is short enough'",
    "really_long_variable_name = 'x' * 60  # this comment pushes the line well past seventy-nine chars",
]

MAX_LEN = 79
for lineno, line in enumerate(sample, start=1):
    if len(line) > MAX_LEN:
        print(f"line {lineno}: E501 line too long ({len(line)} > {MAX_LEN} chars)")
    else:
        print(f"line {lineno}: OK ({len(line)} chars)")

line 1: OK (5 chars)
line 2: OK (37 chars)
line 3: E501 line too long (97 > 79 chars)


---
## Part 3 — Documenting Your Project

### PEP 257 — docstring conventions

**PEP 257** defines the semantics and conventions for **docstrings**.

A **docstring** is a string literal that appears as the **first statement** of a module,
function, class, or method. Python stores it as the object's `__doc__` attribute, and tools
like `help()`, IDEs, and Sphinx read it from there.

**Core rules:**

- Always use **triple double quotes** (three `"` characters) even for one-liners.
- Write a **phrase ending in a period**.
- For functions, **specify the return value**.

In [11]:
def function(a, b):
    """Return the sum of a and b."""
    return a + b


# The docstring is available at runtime as __doc__ ...
print(function.__doc__)
# ... and through the built-in help() system:
help(function)

Return the sum of a and b.
Help on function function in module __main__:

function(a, b)
    Return the sum of a and b.



### One-line vs. multi-line docstrings

A **one-line docstring** fits a simple function. A **multi-line docstring** starts with a
one-line summary, then a blank line, then more detail: arguments, return value, exceptions,
and examples.

In [12]:
def complex_number(real=0.0, imag=0.0):
    """Form a complex number.

    Keyword arguments:
    real -- the real part (default 0.0)
    imag -- the imaginary part (default 0.0)
    """
    return complex(real, imag)


print(complex_number(3, 4))
print(complex_number.__doc__)

(3+4j)
Form a complex number.

Keyword arguments:
real -- the real part (default 0.0)
imag -- the imaginary part (default 0.0)



### Common docstring styles

PEP 257 sets the *rules*; teams then pick a *format*. The three most common, all understood
by Sphinx (with extensions), are shown below. Pick one and use it consistently.

In [13]:
# Google style — readable, popular
def transfer(account: str, amount: float) -> bool:
    """Transfer money to an account.

    Args:
        account: The destination account number.
        amount: The amount to transfer, in euros.

    Returns:
        True if the transfer succeeded, False otherwise.

    Raises:
        ValueError: If amount is negative.
    """
    if amount < 0:
        raise ValueError("amount must be non-negative")
    return True


print(transfer("NL00", 42.0))

True


In [14]:
# NumPy style — common in scientific code
def normalize(values: list[float]) -> list[float]:
    """Scale values to the range [0, 1].

    Parameters
    ----------
    values : list of float
        The numbers to normalize. Must be non-empty.

    Returns
    -------
    list of float
        The values scaled so the minimum maps to 0 and the maximum to 1.
    """
    lo, hi = min(values), max(values)
    span = hi - lo or 1.0
    return [(v - lo) / span for v in values]


print(normalize([10, 20, 30, 40]))

[0.0, 0.3333333333333333, 0.6666666666666666, 1.0]


In [15]:
# reStructuredText / Sphinx style — the classic ":param:" form
def send_email(to: str, subject: str) -> None:
    """Send an email.

    :param to: The recipient's address.
    :param subject: The subject line.
    :returns: Nothing.
    :raises ValueError: If ``to`` is empty.
    """
    if not to:
        raise ValueError("recipient required")


help(send_email)

Help on function send_email in module __main__:

send_email(to: str, subject: str) -> None
    Send an email.

    :param to: The recipient's address.
    :param subject: The subject line.
    :returns: Nothing.
    :raises ValueError: If ``to`` is empty.



### Sphinx — generating documentation from your code

**Sphinx** is the de-facto standard Python documentation generator.

- It reads **reStructuredText** (`.rst`) source and renders **HTML, PDF, and more**.
- With the `autodoc` extension it **extracts docstrings directly from your code**, so your
  API reference stays in sync with the source.

**Typical workflow:**

```bash
pip install sphinx
sphinx-quickstart docs           # scaffold docs/ with conf.py, index.rst, Makefile
# add "sphinx.ext.autodoc" to extensions in docs/conf.py
sphinx-apidoc -o docs/ mypackage # generate .rst stubs from your package
cd docs && make html             # build docs/_build/html/index.html
```

The `demos/` folder in this module contains a worked `docs/` setup (a `gamedemo` package
with `conf.py`, `index.rst`, and a built `_build/html/`) you can inspect as a reference.

> **The payoff:** write one good docstring and it serves three audiences at once — the
> reader of your source, the user of `help()`, and the visitor to your rendered HTML docs.

---
## Part 4 — Error Handling & Exceptions

### try / except basics

Wrap code that can fail in a `try` block and handle specific exception types in `except`
clauses. Catch the **most specific** exception you can.

In [16]:
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("cannot divide by zero")
    except TypeError:
        print("either a or b is not a number")


print(divide(10, 2))
divide(10, 0)
divide(10, "x")

5.0
cannot divide by zero
either a or b is not a number


### Where do exceptions come from?

Exceptions typically occur at **system boundaries** — the places where your program touches
the unpredictable outside world:

- Reading and writing **files**
- Sending data over the **network**
- Taking **user input**
- Reading from or writing to a **database**
- Calling **external programs**
- Calling **code that is not under your control**

Inside pure, self-contained logic, exceptions are rarer. Focus your handling at the edges.

### Three principles for handling exceptions

1. **Only catch exceptions you can meaningfully handle.** Catching an error you can't do
   anything about just hides it.
2. **Sometimes crashing is the correct thing to do.** If the program can't sensibly
   continue, a clear crash (with a traceback for you, the developer) beats limping along in
   a corrupt state.
3. **Handle exceptions in the right layer.** Low-level code detects the problem; a
   higher-level layer that has enough context decides what to do about it.

### EAFP: Easier to Ask Forgiveness than Permission

Python style prefers **trying the operation and handling failure** (EAFP) over **checking
first** (LBYL — Look Before You Leap). The "check first" approach has a race condition: the
file can disappear between the check and the open.

In [17]:
import os

path = "this_file_probably_does_not_exist.txt"

# LBYL — Look Before You Leap (race-prone, less Pythonic)
if os.path.exists(path):
    with open(path) as f:
        data = f.read()
    print("LBYL read the file")
else:
    print("LBYL: file does not exist")

LBYL: file does not exist


In [18]:
# EAFP — Easier to Ask Forgiveness than Permission (preferred)
# Bonus: use a context manager (with-block) so the file is always closed.
try:
    with open(path) as f:
        data = f.read()
    print("EAFP read the file")
except FileNotFoundError:
    print("EAFP: handled missing file cleanly")

EAFP: handled missing file cleanly


### The "happy path" pattern

EAFP lets you write the **happy flow** as a straight line inside `try`, with the error cases
handled separately below. Compare this store example (adapted from the course demos) to a
version cluttered with `if` checks.

In [19]:
products = ["bread", "wine", "coffee"]


def select_product(raw_choice: str):
    try:
        # Just do the work: the "happy flow"
        choice = int(raw_choice)
        product = products[choice]
        print(f"Selected: {product}")
    except ValueError:
        print("That is not a number")
    except IndexError:
        print("Invalid product number")


select_product("1")     # happy path
select_product("nope")  # ValueError
select_product("9")     # IndexError

Selected: wine
That is not a number
Invalid product number


### Never do this: swallowing bare exceptions

Both anti-patterns below hide *what* went wrong and *where*:

```python
try:
    # code here...
except Exception:                     # too broad
    print("something went wrong")

try:
    # code here...
except:                               # bare except — even worse
    print("something went wrong")
```

A **bare `except:`** even catches `KeyboardInterrupt` and `SystemExit`, so you can't Ctrl-C
out of the program. Catching `Exception` and printing a generic message throws away the
traceback you need to debug.

**What to do instead:**

- **Logging and re-raising is always okay** — record the error, then let it propagate.
- **Never log sensitive data** (passwords, tokens, personal data).
- **Don't show tracebacks or internal details on public pages** — they leak information to
  attackers. Show users a friendly message; log the details privately.
- **Document the exceptions your code can raise.**

In [20]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("demo")


def risky_operation(x):
    return 10 / x


# GOOD: log with context, then re-raise so callers still see the failure.
try:
    risky_operation(0)
except ZeroDivisionError:
    log.exception("risky_operation failed for x=0")   # logs traceback
    # In real code you might: raise   (re-raise to the caller)
    print("Handled: logged the error, could re-raise here.")

ERROR: risky_operation failed for x=0
Traceback (most recent call last):
  File "/var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/ipykernel_19321/1487986979.py", line 13, in <module>
    risky_operation(0)
    ~~~~~~~~~~~~~~~^^^
  File "/var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/ipykernel_19321/1487986979.py", line 8, in risky_operation
    return 10 / x
           ~~~^~~
ZeroDivisionError: division by zero


Handled: logged the error, could re-raise here.


### Raise an appropriate exception type

When *you* detect a problem, raise the exception that best describes it. Python's built-in
types already cover most cases:

- **`TypeError`** — an argument is of the wrong *type*.
- **`ValueError`** — the type is right but the *value* is invalid.
- **`KeyError` / `IndexError`** — missing key or out-of-range index.

In [21]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an int")
    if age < 0:
        raise ValueError("age must be non-negative")
    return age


for candidate in [30, -5, "old"]:
    try:
        print("OK:", set_age(candidate))
    except (TypeError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

OK: 30
ValueError: age must be non-negative
TypeError: age must be an int


### Add a helpful message

The exception message is often the first thing a developer sees. Make it actionable —
include the offending value using an f-string.

In [22]:
def new_user(username):
    if len(username) < 3:
        raise ValueError(f"username too short: {username!r}")
    return username


try:
    new_user("ab")
except ValueError as e:
    print("Error:", e)

Error: username too short: 'ab'


### Custom exception classes

Define your own exception types to model your domain. A custom exception is often just an
empty subclass of `Exception`. You can build a **hierarchy** so callers catch at whatever
level of granularity they need.

In [23]:
class BillingFailedException(Exception):
    """Base class for anything that goes wrong during billing."""


class CreditCardDeclinedException(BillingFailedException):
    pass


class NoBillingInfoException(BillingFailedException):
    pass


def charge(card_ok: bool, has_info: bool):
    if not has_info:
        raise NoBillingInfoException("no billing info on file")
    if not card_ok:
        raise CreditCardDeclinedException("card was declined")
    return "charged"


# A caller can catch a SPECIFIC failure...
try:
    charge(card_ok=False, has_info=True)
except CreditCardDeclinedException as e:
    print("Specific handler:", e)

# ...or catch the WHOLE family via the base class.
try:
    charge(card_ok=True, has_info=False)
except BillingFailedException as e:
    print("Family handler:", type(e).__name__, "-", e)

Specific handler: card was declined
Family handler: NoBillingInfoException - no billing info on file


### Exception chaining with `raise ... from`

When you catch a low-level exception and raise your own domain exception, use
**`raise NewError(...) from original`**. This preserves the original cause in the traceback
(shown as *"The above exception was the direct cause of the following exception"*), which is
invaluable for debugging.

In [24]:
class AdditionError(Exception):
    pass


def plus(a, b):
    try:
        return a + b
    except TypeError as e:
        raise AdditionError("Error occurred while adding numbers.") from e


try:
    plus(2, "3")
except AdditionError as e:
    print("Caught:", e)
    print("Original cause:", repr(e.__cause__))

Caught: Error occurred while adding numbers.
Original cause: TypeError("unsupported operand type(s) for +: 'int' and 'str'")


### Handling exceptions in the right layer

This is where all the principles come together. The course demo splits a user-management
app into layers:

- **DB layer** — only stores and retrieves users. It raises a low-level
  `UniqueConstraintError` and knows nothing about the UI.
- **Business logic layer** — orchestrates billing, DB, and email. It *decides what to do*
  when the DB reports a conflict.
- **UI layer** — talks to the user. It turns exceptions into friendly messages.

Each layer catches only what it has the context to handle, and lets everything else
propagate upward.

In [25]:
# --- DB layer: storage only, no UI, no business rules ---
class DB:
    class UniqueConstraintError(Exception):
        pass

    def __init__(self):
        self.users: set[str] = set()

    def insert(self, username: str):
        if username in self.users:
            raise self.UniqueConstraintError(f"user {username} already exists.")
        self.users.add(username)

    def list_users(self) -> set[str]:
        return self.users


# --- Business logic layer: decides HOW to react to a low-level error ---
def create_user(db: DB, username: str):
    try:
        db.insert(username)
    except DB.UniqueConstraintError:
        # Business rule: on a clash, append a suffix and retry.
        db.insert(username + "_1")


# --- UI layer: turns outcomes into user-facing messages ---
def create_user_ui(db: DB, username: str):
    try:
        create_user(db, username)
        print(f"User {username!r} created.")
    except DB.UniqueConstraintError:
        print("Sorry, that username is already taken.")


db = DB()
create_user_ui(db, "alice")
create_user_ui(db, "alice")   # clash -> business layer retries as alice_1
print("Users now:", sorted(db.list_users()))

User 'alice' created.
User 'alice' created.
Users now: ['alice', 'alice_1']


---
## Part 5 — Effective Code Reviews

### The setup

Code reviews assume a certain way of working:

- **Teamwork**, with tasks assigned to members.
- **Changes** made in a shared codebase.
- Code **stored in Git** (or a similar VCS).
- Changes **shared through a Pull Request** (or merge request) before they land.

### Why do we have code reviews?

- **Quality** — prevent errors, improve design, and keep the code maintainable.
- **Sharing knowledge** — reviews spread understanding of the codebase across the team, so
  no single person is the only one who knows an area.

> **It's not personal — so try to be polite.** A review critiques *the code*, not the
> author. Being kind and constructive keeps the practice healthy and the team learning.

### The Code Review Pyramid

Not all review comments carry the same weight. The **Code Review Pyramid** orders concerns
from most important (bottom) to least (top). Spend your attention where it matters most, and
**automate the top layers** so humans don't waste review time on them.

```
                    Style                     <- automate (ruff/black)
                   Testing                    <- partly automate (CI, coverage)
                Documentation
           Behaviour and Correctness
          Design and Architecture             <- most important, hardest to automate
```

The lower a concern sits, the more expensive it is to fix later and the harder it is to
catch automatically. A perfectly formatted function that solves the wrong problem is still
worthless.

### Layer 1 (bottom): Design and Architecture — *most important*

Start at the foundation. Before checking any detail, ask whether the change makes sense at all:

- **Which problem are we solving?** Is this the correct solution?
- Does it do **what was asked — no more, no less**? (YAGNI applies to reviews too.)
- **Does the API make sense?** Are the responsibilities of classes and functions clear? Are the arguments and return types sensible?
- Is it **consistent** with the rest of the codebase?
- **Does it break existing code?**

Getting this layer wrong is the most expensive mistake, because it's the hardest to undo.

### Layer 2: Behaviour and Correctness

Once the design is sound, verify the implementation actually works:

- Is it **logically correct**?
- Is the **complexity** appropriate?
- **Performance** — acceptable for the use case?
- **Error handling** — are failures handled in the right layer (see Part 4)?
- **Security** — any injection, secret-leakage, or auth issues?
- **Observability** — can you tell what happened in production (logs, metrics)?

> **Give concrete examples when you see problems.** "This breaks when `items` is empty:
> `total()` divides by `len(items)`" is far more useful than "this looks buggy."

### Layer 3: Documentation

- Are all **public components documented**? Are the docs **clear, readable**, and do they
  **contain code examples**?
- **Do the docs match the implementation?** Stale docs are worse than none — they actively
  mislead.

### Layer 4: Testing

- Is **all business logic tested**, including **edge cases**, **failure scenarios**, and
  **non-functional** aspects?
- Have tests been **updated for changed code**?
- Were **tests added for fixed bugs** (so they never regress)?
- Do **all tests run and succeed**? (Automate this in CI.)
- Is **code coverage** tracked? (Automate this too.)

> **Hard-to-test functions are a code smell.** If something is painful to test, that usually
> points back to a design problem in Layer 1 — tight coupling or a missing seam. The
> difficulty is a signal, not just an inconvenience.

### Layer 5 (top): Style — *automate it*

Style is the *least* important thing to spend human review time on, precisely because
machines do it better:

- **Use an autoformatting tool** like **black** or **ruff**.
- **Run linters, formatters, and unit tests BEFORE the code review** — ideally in a
  pre-commit hook and in CI.

If the machine has already enforced style, reviewers never argue about spaces vs. tabs and
can focus on design and correctness.

### Communication: be kind and constructive

The single most important soft skill in reviews:

- **Try to be kind and constructive** in your communication.
- Critique the code, not the person.
- Prefer questions and suggestions ("What do you think about extracting this?") over
  commands.
- **Give concrete examples** for every problem you raise.

A code review is a conversation between colleagues who both want the code to be good.

In [26]:
# A miniature "review pyramid" checklist you could run before opening a PR.
checklist = {
    "Design & Architecture": [
        "Solves the actual problem, nothing extra (YAGNI)",
        "API / responsibilities make sense",
        "Consistent with the codebase; doesn't break existing code",
    ],
    "Behaviour & Correctness": [
        "Logically correct, edge cases covered",
        "Errors handled in the right layer",
        "No obvious security or performance issues",
    ],
    "Documentation": [
        "Public components documented",
        "Docs match the implementation",
    ],
    "Testing": [
        "Business logic + edge cases + failure scenarios tested",
        "Tests added for fixed bugs; all tests pass",
    ],
    "Style (automate!)": [
        "ruff / black run clean",
        "Linters + tests run before review (pre-commit / CI)",
    ],
}

for layer, items in checklist.items():
    print(f"\n{layer}")
    for item in items:
        print(f"  [ ] {item}")


Design & Architecture
  [ ] Solves the actual problem, nothing extra (YAGNI)
  [ ] API / responsibilities make sense
  [ ] Consistent with the codebase; doesn't break existing code

Behaviour & Correctness
  [ ] Logically correct, edge cases covered
  [ ] Errors handled in the right layer
  [ ] No obvious security or performance issues

Documentation
  [ ] Public components documented
  [ ] Docs match the implementation

Testing
  [ ] Business logic + edge cases + failure scenarios tested
  [ ] Tests added for fixed bugs; all tests pass

Style (automate!)
  [ ] ruff / black run clean
  [ ] Linters + tests run before review (pre-commit / CI)


---
## Summary — the whole path on one page

| Module | The one thing to remember |
|--------|---------------------------|
| **1. Maintainable code** | Aim for **high cohesion, loose coupling**; keep it simple (KISS) and don't build what you don't need (YAGNI). |
| **2. PEP 8 style** | Consistency serves the reader. Let **ruff/black** enforce it automatically. |
| **3. Documentation** | Write **docstrings** (PEP 257) for everything public; generate HTML docs with **Sphinx**. |
| **4. Error handling** | Catch only what you can handle, **at the right layer**; prefer **EAFP**; chain with `raise ... from`. |
| **5. Code reviews** | Follow the **pyramid** (design > correctness > docs > tests > style); automate the top, be kind at every layer. |

### How the modules connect

These aren't five separate topics — they reinforce each other:

- **Loose coupling** (Module 1) is what makes code **testable** (a Module 5 concern) and
  makes errors easy to **handle in the right layer** (Module 4).
- **Docstrings** (Module 3) are mandated by **PEP 8** (Module 2) and checked during the
  **documentation** layer of a review (Module 5).
- **Style tooling** (Module 2) is exactly what lets you **automate the top of the review
  pyramid** (Module 5).

### Practice ideas

1. Take a function you wrote recently. Can you explain its responsibility in one sentence? If not, split it.
2. Run `ruff check` and `ruff format` on a small project and read every warning.
3. Add Google-style docstrings to one module and build the HTML with Sphinx.
4. Find a bare `except:` in any codebase and replace it with a specific handler that logs and re-raises.
5. Review a teammate's (or your own) recent PR using the pyramid checklist above.

### References

- *Clean Code* — Robert C. Martin
- [PEP 8 — Style Guide for Python Code](https://peps.python.org/pep-0008/)
- [PEP 257 — Docstring Conventions](https://peps.python.org/pep-0257/)
- [PEP 20 — The Zen of Python](https://peps.python.org/pep-0020/) (`import this`)
- [ruff](https://docs.astral.sh/ruff/) · [black](https://black.readthedocs.io/) · [Sphinx](https://www.sphinx-doc.org/)
- [The Code Review Pyramid](https://www.morling.dev/blog/the-code-review-pyramid/) — Gunnar Morling
- Course by Reindert-Jan Ekker — [www.codesensei.nl](https://www.codesensei.nl)

In [27]:
# The Zen of Python — a fitting closing note. Run it!
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!
